In [1]:
from pathlib import Path
import ast

import pandas as pd
from datasets import load_dataset

In [2]:
from pprint import pprint
import os

In [3]:
data_dir = Path(os.getcwd()).resolve().parent/"data"
data_dir

PosixPath('/home/zelluzy/Desktop/recipe-genome/data')

In [ ]:
ds = load_dataset("untitledwebsite123/food-recipes")

In [5]:
print(repr(ds))

DatasetDict({
    train: Dataset({
        features: ['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions'],
        num_rows: 522517
    })
})


In [6]:
ds_ = ds
ds = ds_["train"]

In [7]:
ds.features

{'RecipeId': Value('int64'),
 'Name': Value('string'),
 'AuthorId': Value('int64'),
 'AuthorName': Value('string'),
 'CookTime': Value('string'),
 'PrepTime': Value('string'),
 'TotalTime': Value('string'),
 'DatePublished': Value('string'),
 'Description': Value('string'),
 'Images': Value('string'),
 'RecipeCategory': Value('string'),
 'Keywords': Value('string'),
 'RecipeIngredientQuantities': Value('string'),
 'RecipeIngredientParts': Value('string'),
 'AggregatedRating': Value('float64'),
 'ReviewCount': Value('float64'),
 'Calories': Value('float64'),
 'FatContent': Value('float64'),
 'SaturatedFatContent': Value('float64'),
 'CholesterolContent': Value('float64'),
 'SodiumContent': Value('float64'),
 'CarbohydrateContent': Value('float64'),
 'FiberContent': Value('float64'),
 'SugarContent': Value('float64'),
 'ProteinContent': Value('float64'),
 'RecipeServings': Value('float64'),
 'RecipeYield': Value('string'),
 'RecipeInstructions': Value('string')}

In [8]:
ds[0]

{'RecipeId': 38,
 'Name': 'Low-Fat Berry Blue Frozen Dessert',
 'AuthorId': 1533,
 'AuthorName': 'Dancer',
 'CookTime': 'PT24H',
 'PrepTime': 'PT45M',
 'TotalTime': 'PT24H45M',
 'DatePublished': '1999-08-09T21:46:00Z',
 'Description': 'Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.',
 'Images': 'c("https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg", \n"https://img.sndimg.com/food/image/upload/w_555,h_416,c

In [9]:
ast.literal_eval(ds[0]["RecipeInstructions"].replace("c", "", count=1))

('Toss 2 cups berries with sugar.',
 'Let stand for 45 minutes, stirring occasionally.',
 'Transfer berry-sugar mixture to food processor.',
 'Add yogurt and process until smooth.',
 "Strain through fine sieve. Pour into baking pan (or transfer to ice cream maker and process according to manufacturers' directions). Freeze uncovered until edges are solid but centre is soft.  Transfer to processor and blend until smooth again.",
 'Return to pan and freeze until edges are solid.',
 'Transfer to processor and blend until smooth again.',
 'Fold in remaining 2 cups of blueberries.',
 'Pour into plastic mold and freeze overnight. Let soften slightly to serve.')

In [10]:
format_columns = [
    "Images", 
    "Keywords", 
    "RecipeIngredientParts",
    "RecipeIngredientQuantities",
    "RecipeInstructions",
]

In [11]:
sub = ds.select(range(100))

In [12]:
i = 0
for ex in sub:
    i+= 1
    for cat in format_columns:
        if not isinstance(ex[cat], str):
            continue
        
        if ex[cat].startswith("c"):
            c = ex[cat].replace("c", "", 1)
        else:
            c = ex[cat]
            
        if "NA" in c:
            c = c.replace("NA", "None")
            
        if "haracter" in c.lower():
            continue
        # try:
        # pprint(list(ast.literal_eval(c)))
        # except ValueError as e:
        #     print("=" * 100)
        #     print(f"error: {e}")
        #     print(str(c))
        #     print("=" * 100)
    # print(i)

In [14]:
import re
from typing import Any

def parse(cell: Any) -> list[Any]:
    """parses ds cell into python list. 
    if v is not a str an empty list is returned

    Args:
        cell (Any): cell to parse

    Returns:
        list[Any]: parsed cell as a list
    """
    if not isinstance(cell, str) or not cell:
        return []
    if cell.startswith("c"):
        cell = cell[1:]
    # use re with word boundary for words like "banana"
    cell = re.sub(r"\bNA\b", "None", cell)
    if "haracter" in cell:
        return []
    try:
        cell = ast.literal_eval(cell)
    except SyntaxError:
        pass
    return list(cell) if isinstance(cell, tuple) else [cell]


In [ ]:
from datetime import timedelta
abs(timedelta(minutes=-100))

In [ ]:
from datetime import timedelta

# CookTime/PrepTime/TotalTime are ISO 8601 *durations* (e.g. "PT24H45M"),
# not datetimes, so datetime.fromisoformat doesn't apply here.
# some rows have negative components (e.g. "PT-30M"), hence the -? in each group.
ISO_DURATION_RE = re.compile(
    r"^P(?:(?P<days>-?\d+)D)?"
    r"(?:T(?:(?P<hours>-?\d+)H)?(?:(?P<minutes>-?\d+)M)?(?:(?P<seconds>-?\d+)S)?)?$"
)

def parse_duration(s: str) -> timedelta:
    if not s:
        return timedelta(seconds=0)
    m = ISO_DURATION_RE.match(s)
    if not m:
        raise ValueError(f"Invalid ISO 8601 duration: {s!r}")
    return abs(timedelta(**{k: int(v) for k, v in m.groupdict().items() if v}))

times = ["CookTime", "PrepTime", "TotalTime"]

formatted = {ds[0][col]:parse_duration(ds[0][col]) for col in times}
formatted


{'PT24H': datetime.timedelta(days=1),
 'PT45M': datetime.timedelta(seconds=2700),
 'PT24H45M': datetime.timedelta(days=1, seconds=2700)}

In [34]:
for ex in sub:
    for cat in times:
        p = parse_duration(ex[cat])
        print(p)

1 day, 0:00:00
0:45:00
1 day, 0:45:00
0:25:00
4:00:00
4:25:00
0:05:00
0:30:00
0:35:00
0:20:00
1 day, 0:00:00
1 day, 0:20:00
0:30:00
0:20:00
0:50:00
2:00:00
0:20:00
2:20:00
0:03:00
0:35:00
0:38:00
0:50:00
0:30:00
1:20:00
None
0:25:00
0:25:00
0:09:00
0:55:00
1:04:00
None
2:15:00
2:15:00
0:30:00
0:45:00
1:15:00
0:50:00
0:20:00
1:10:00
0:25:00
0:15:00
0:40:00
None
0:05:00
0:05:00
0:45:00
1:05:00
1:50:00
0:50:00
0:45:00
1:35:00
2:00:00
0:05:00
2:05:00
1:00:00
0:20:00
1:20:00
None
0:10:00
0:10:00
2:14:00
0:30:00
2:44:00
0:10:00
0:30:00
0:40:00
None
0:35:00
0:35:00
0:42:00
0:35:00
1:17:00
0:15:00
0:10:00
0:25:00
0:25:00
0:15:00
0:40:00
1:00:00
0:15:00
1:15:00
0:20:00
0:10:00
0:30:00
1:00:00
1:00:00
2:00:00
2:38:00
0:35:00
3:13:00
1:50:00
2:45:00
4:35:00
3:00:00
0:35:00
3:35:00
0:43:00
1:25:00
2:08:00
0:35:00
0:40:00
1:15:00
1:00:00
0:50:00
1:50:00
0:55:00
0:35:00
1:30:00
None
0:25:00
0:25:00
0:10:00
0:05:00
0:15:00
None
0:20:00
0:20:00
3:00:00
0:25:00
3:25:00
0:45:00
0:20:00
1:05:00
0:30:00
0

In [35]:
for col in format_columns:
    for item in ds.select(range(5))[col]:
        print(parse(item))

['https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg', 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg', 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg', 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg', 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/picuaETeN.jpg', 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/pictzvxW5.jpg']
['https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picM9Mhnw.jpg', 'https://img.sndimg.com/food/image/upload

In [ ]:
import os
def format_batch(batch: dict[str, list[Any]], reg_columns: list[str], 
                 dur_columns: list[str]):
    reg = {col: [parse(v) for v in batch[col]] for col in reg_columns}
    dur = {col: [parse_duration(v) for v in batch[col]] for col in dur_columns}
    
    return reg | dur

In [37]:
sub.map(
    lambda b: format_batch(b, format_columns, dur_colums=times), 
    batched=True,  
    num_proc=os.cpu_count()-1
)

Map (num_proc=11):   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions'],
    num_rows: 100
})

In [38]:
parsed_ds = ds.map(
    lambda b: format_batch(b, format_columns, dur_colums=times), 
    batched=True, 
    batch_size=100_000, 
    num_proc=os.cpu_count()-1
)

Map (num_proc=11):   0%|          | 0/522517 [00:00<?, ? examples/s]

In [39]:
parsed_ds[0]

{'RecipeId': 38,
 'Name': 'Low-Fat Berry Blue Frozen Dessert',
 'AuthorId': 1533,
 'AuthorName': 'Dancer',
 'CookTime': datetime.timedelta(days=1),
 'PrepTime': datetime.timedelta(seconds=2700),
 'TotalTime': datetime.timedelta(days=1, seconds=2700),
 'DatePublished': '1999-08-09T21:46:00Z',
 'Description': 'Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.',
 'Images': ['https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg',
  'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg',
  'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg',
  'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_2

In [40]:
parsed_ds.save_to_disk(data_dir/ "parsed-recipes")

Saving the dataset (0/2 shards):   0%|          | 0/522517 [00:00<?, ? examples/s]